## Final Project Submission

Please fill out:
* Student name: Oliver Kiplagat.
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: Samuel G.Mwangi.
* Blog post URL:


##  EDA: Financial Analysis — Movie Budgets Dataset

### **Objective**
Load, inspect, and perform initial structural analysis on **The Numbers Movie Budgets** dataset (`tn.movie_budgets.csv.gz`) to evaluate movie production budgets, domestic gross revenue, and worldwide gross revenue for ROI calculations.

This dataset is the only one that tells us what a movie actually **cost** to make.
Every other dataset in this project tells us how much money came *in* — this one
tells us how much went *out*. Without it, we can't calculate profit or return on
investment, which makes it one of the most important pieces of our analysis.

We'll load it, look at its shape and structure, and check for anything that needs
cleaning before we can use it.

In [14]:
# --- Core data libraries ---
import pandas as pd
import numpy as np

# --- Visualization libraries ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Database and file handling ---
import sqlite3
import zipfile
import re
from pathlib import Path

# --- Display settings: show full dataframes, not truncated ---
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# --- Reliable, break-proof file paths ---
# Path(__file__).parent would be used in a .py script, but in a notebook we use
# Path.cwd() (current working directory) as our anchor point instead.
PROJECT_ROOT = Path.cwd()
ZIPPED_DATA_DIR = PROJECT_ROOT / "zippedData"
DATA_DIR = PROJECT_ROOT / "data"

# Make sure the data folder exists before we try to unzip anything into it
DATA_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Zipped data folder:", ZIPPED_DATA_DIR)
print("Zipped data folder exists:", ZIPPED_DATA_DIR.exists())

Project root: /home/komen/Documents/moringa_school_projects/Module_3_labs/Group_2_project/Studio_Launch-Pad
Zipped data folder: /home/komen/Documents/moringa_school_projects/Module_3_labs/Group_2_project/Studio_Launch-Pad/zippedData
Zipped data folder exists: True


In [18]:
tn_budgets_path = ZIPPED_DATA_DIR / "tn.movie_budgets.csv.gz"

# Confirm the file actually exists before attempting to load it
assert tn_budgets_path.exists(), f"File not found: {tn_budgets_path}"

df_budgets = pd.read_csv(tn_budgets_path)

print("Loaded successfully.")
print("Shape (rows, columns):", df_budgets.shape)

Loaded successfully.
Shape (rows, columns): (5782, 6)


In [17]:
#Data Inspection.
print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"{df_budgets.shape[0]} rows, {df_budgets.shape[1]} columns")

print()
print("=" * 60)
print("COLUMN INFO (names, types, non-null counts)")
print("=" * 60)
df_budgets.info()

print()
print("=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
display(df_budgets.head())

print()
print("=" * 60)
print("LAST 5 ROWS")
print("=" * 60)
display(df_budgets.tail())

print()
print("=" * 60)
print("MISSING VALUES PER COLUMN")
print("=" * 60)
print(df_budgets.isna().sum())

print()
print("=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)
print("Exact duplicate rows:", df_budgets.duplicated().sum())
print("Duplicate movie + release_date combos:", df_budgets.duplicated(subset=["movie", "release_date"]).sum())

print()
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
df_budgets.describe(include="all")

SHAPE
5782 rows, 6 columns

COLUMN INFO (names, types, non-null counts)
<class 'pandas.DataFrame'>
RangeIndex: 5782 entries, 0 to 5781
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 5782 non-null   int64
 1   release_date       5782 non-null   str  
 2   movie              5782 non-null   str  
 3   production_budget  5782 non-null   str  
 4   domestic_gross     5782 non-null   str  
 5   worldwide_gross    5782 non-null   str  
dtypes: int64(1), str(5)
memory usage: 595.5 KB

FIRST 5 ROWS


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"



LAST 5 ROWS


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
5777,78,"Dec 31, 2018",Red 11,"$7,000",$0,$0
5778,79,"Apr 2, 1999",Following,"$6,000","$48,482","$240,495"
5779,80,"Jul 13, 2005",Return to the Land of Wonders,"$5,000","$1,338","$1,338"
5780,81,"Sep 29, 2015",A Plague So Pleasant,"$1,400",$0,$0
5781,82,"Aug 5, 2005",My Date With Drew,"$1,100","$181,041","$181,041"



MISSING VALUES PER COLUMN
id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

DUPLICATE ROWS
Exact duplicate rows: 0
Duplicate movie + release_date combos: 0

SUMMARY STATISTICS


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
count,5782.000000,5782,5782,5782,5782,5782
unique,NaN,2418,5698,509,5164,5356
top,NaN,"Dec 31, 2014",King Kong,"$20,000,000",$0,$0
freq,NaN,24,3,231,548,367
mean,50.372363,NaN,NaN,NaN,NaN,NaN
std,28.821076,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN
25%,25.000000,NaN,NaN,NaN,NaN,NaN
50%,50.000000,NaN,NaN,NaN,NaN,NaN
75%,75.000000,NaN,NaN,NaN,NaN,NaN


## Cleaning Data Types

Looking back at what we found earlier , the money columns and the release date
are technically there, but not really usable yet. `production_budget`,
`domestic_gross`, and `worldwide_gross` are all stored as plain text because of
the dollar signs and commas (something like `"$425,000,000"`), which means we
can't do any real math on them until that's fixed. `release_date` has the same
kind of problem ; it's just a string right now, not something pandas actually
understands as a date.

We fix both here: strip the `$` and `,` out of the money columns and convert
them to numbers, and convert `release_date` into a proper datetime so we can
pull the release year straight out of it. 

In [19]:
# Money columns are still text because of the $ and commas, so let's fix that

money_cols = ["production_budget", "domestic_gross", "worldwide_gross"]

for col in money_cols:
    # strip out $ and , then convert to a proper number
    df_budgets[col] = df_budgets[col].replace("[\$,]", "", regex=True).astype(float)

# quick check - did it actually work? should say float64 now, not object
print(df_budgets[money_cols].dtypes)

# fixing release_date which is a string.
# turning it into an actual date.
df_budgets["release_date"] = pd.to_datetime(df_budgets["release_date"])
df_budgets["year"] = df_budgets["release_date"].dt.year

df_budgets.head()

production_budget    float64
domestic_gross       float64
worldwide_gross      float64
dtype: object


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross,year
0,1,2009-12-18,Avatar,425000000.0,760507625.0,2.776345e+09,2009
1,2,2011-05-20,Pirates of the Caribbean: On Stranger Tides,410600000.0,241063875.0,1.045664e+09,2011
2,3,2019-06-07,Dark Phoenix,350000000.0,42762350.0,1.497624e+08,2019
3,4,2015-05-01,Avengers: Age of Ultron,330600000.0,459005868.0,1.403014e+09,2015
4,5,2017-12-15,Star Wars Ep. VIII: The Last Jedi,317000000.0,620181382.0,1.316722e+09,2017


In [ ]:
# standardize the title so it can actually match up with the other datasets later
def clean_title(title):
    title = str(title).lower().strip()
    title = re.sub(r"[^\w\s]", "", title)
    return title

df_budgets["clean_title"] = df_budgets["movie"].apply(clean_title)

# now the numbers that actually matter for the business questions -
# profit is just what came in minus what went out
df_budgets["profit"] = df_budgets["worldwide_gross"] - df_budgets["production_budget"]

# roi(return of investments) tells us how many times over the budget was earned back
# guarding against divide-by-zero just in case any budget is 0
df_budgets["roi"] = df_budgets["profit"] / df_budgets["production_budget"].replace(0, np.nan)

# quick look - does this actually make sense?
display(df_budgets[["movie", "clean_title", "production_budget", "worldwide_gross", "profit", "roi"]].head())

# sanity check on the numbers overall
print(df_budgets[["profit", "roi"]].describe())

,movie,clean_title,production_budget,worldwide_gross,profit,roi
0,Avatar,avatar,425000000.0,2.776345e+09,2.351345e+09,5.532577
1,Pirates of the Caribbean: On Stranger Tides,pirates of the caribbean on stranger tides,410600000.0,1.045664e+09,6.350639e+08,1.546673
2,Dark Phoenix,dark phoenix,350000000.0,1.497624e+08,-2.002376e+08,-0.572108
3,Avengers: Age of Ultron,avengers age of ultron,330600000.0,1.403014e+09,1.072414e+09,3.243841
4,Star Wars Ep. VIII: The Last Jedi,star wars ep viii the last jedi,317000000.0,1.316722e+09,9.997217e+08,3.153696


             profit          roi
count  5.782000e+03  5782.000000
mean   5.989970e+07     3.800161
std    1.460889e+08    29.530282
min   -2.002376e+08    -1.000000
25%   -2.189071e+06    -0.507704
50%    8.550286e+06     0.708310
75%    6.096850e+07     2.758346
max    2.351345e+09  1799.000000
